# AmazingHand — điều khiển 4 ngón

Chọn kernel **AmazingHand (.venv)**. Chạy cell **Khởi tạo** một lần, sau đó chọn cell của tư thế bạn muốn và nhấn **Shift+Enter**. Chạy từng tư thế, chờ cell hoàn thành rồi chọn tư thế khác; **Run All** sẽ chạy liên tiếp mọi tư thế trong notebook.

Notebook điều khiển bàn tay thật qua USB, không dùng MuJoCo. Mỗi thao tác tự mở và đóng COM3; sau khi hoàn thành, servo vẫn giữ tư thế. Tốc độ đặt tối đa là 40°/giây; quỹ đạo tự chờ khi servo chưa bắt kịp. Thời gian thực tế phụ thuộc phản hồi servo và thời gian xác nhận tư thế. Cell cuối dùng để nhả lực giữ.

| Ngón | Tên trong code | Servo ID |
| --- | --- | --- |
| Trỏ | index | 1, 2 |
| Giữa | middle | 3, 4 |
| Áp út | ring | 5, 6 |
| Cái | thumb | 7, 8 |


## Khởi tạo — đọc kết nối và trạng thái

Cell này chỉ đọc trạng thái. Đổi `PORT` nếu Windows đổi cổng USB. `DRY_RUN=True` giúp kiểm tra kết nối và mục tiêu mà không gửi lệnh chuyển động; vẫn cần kết nối bàn tay.

Chỉnh `SPEED_DEG_S` để thay đổi tốc độ cho cả bảy tư thế, rồi chạy lại cell khởi tạo. Mặc định là 40°/giây; đặt 12 để quay về mức trước hoặc 4 để dùng mức ban đầu. Cell tự nạp lại các helper từ đúng thư mục project mỗi lần chạy. Các cell thử riêng vẫn dùng 4°/giây.

Đây là tốc độ tối đa yêu cầu. Khi thấy `slowing trajectory to let servos catch up`, chương trình đang chờ phản hồi để servo bắt kịp; thao tác chưa bị lỗi. Nếu servo đứng yên khi còn mục tiêu cần tới hoặc mất phản hồi, chương trình vẫn dừng và thử nhả lực giữ.

`OPEN_EXTENSIONS` và `CLOSE_FLEXIONS` lần lượt đặt mức duỗi và co riêng cho index, middle, ring, thumb, cộng với offset trong project. Mức duỗi mới là **67°, 58°, 37°, 35°**; mức co là **92°, 90°, 90°, 90°**. Index và middle đã được mở tăng dần theo phản hồi; người dùng xác nhận index thẳng ở 62° và middle thẳng ở 58°. Mức co được giữ ở 92°/90°/90°/90°. Kết quả từng cell chạy thật được lưu trong NOTEBOOK_CHECK_VI.md. Đây là góc servo, không phải góc khớp; không tự chạy dò giới hạn cơ khí.

Phản hồi bao gồm góc đặt, góc đo, sai lệch và tải thô. Sau khi thay file, mở lại notebook và chạy cell này để nạp code mới. Mỗi tư thế chờ ổn định tối đa 3 giây, cần 3 mẫu liên tiếp trong sai số 4°; trong lúc chờ vẫn kiểm tra tải, nhiệt độ, điện áp và sai lệch chuyển động 8°. Index được tăng riêng thêm 5° theo yêu cầu; middle giữ 58°.

`ALLOW_VOLTAGE_WARNING=True` dùng chế độ đã thử trước đó: chấp nhận riêng cờ điện áp trong khoảng 4–7,4 V. Kiểm tra tải, nhiệt độ, các lỗi khác và phản hồi vị trí vẫn hoạt động; không sửa cấu hình EEPROM.


`MOTION_PROFILE="smooth"` dùng quỹ đạo tăng/giảm tốc êm, cập nhật mục tiêu với nhịp chờ 20 ms. Phản hồi vị trí/tải/điện áp/nhiệt độ/cờ lỗi được đọc gộp mỗi servo. `SPEED_DEG_S=40` là tốc độ đỉnh yêu cầu; quỹ đạo tự chờ nếu servo chậm. Muốn so với bản trước, đặt `MOTION_PROFILE="linear"`, `SPEED_DEG_S=20` rồi chạy lại cell này. Các góc co/duỗi và ngưỡng xác nhận 4° giữ nguyên.

In [1]:
from pathlib import Path
import sys
import importlib.util

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "PythonExample" / "AmazingHand_Notebook.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Hãy mở notebook từ thư mục AmazingHand hoặc PythonExample.")
print(f"Python đang dùng: {sys.executable}")
if importlib.util.find_spec("rustypot") is None:
    raise RuntimeError(
        "Kernel hiện tại chưa có rustypot. Chọn kernel AmazingHand (.venv) "
        "hoặc Select Kernel > Python Environments > "
        + str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")
        + ", rồi chạy lại cell này."
    )
helper_path = str(PROJECT_ROOT / "PythonExample")
if helper_path in sys.path:
    sys.path.remove(helper_path)
sys.path.insert(0, helper_path)

# Nạp lại theo thứ tự phụ thuộc để kernel không giữ class/hàm của bản cũ.
importlib.invalidate_caches()
for module_name in (
    "AmazingHand_HardwareDemo",
    "AmazingHand_Motion",
    "AmazingHand_GripCycles",
    "AmazingHand_VSign",
    "AmazingHand_Notebook",
):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
    else:
        importlib.import_module(module_name)
notebook_module = sys.modules["AmazingHand_Notebook"]
NotebookHand = notebook_module.NotebookHand
print(f"Helper đang dùng: {notebook_module.__file__}")

PORT = 'COM3'
SPEED_DEG_S = 40
OPEN_EXTENSIONS = (67, 58, 37, 35)
CLOSE_FLEXIONS = (92, 90, 90, 90)
END_TOLERANCE_DEG = 4
MOTION_PROFILE = "smooth"
DRY_RUN = False
ALLOW_VOLTAGE_WARNING = True

hand = NotebookHand(
    port=PORT,
    speed_deg_s=SPEED_DEG_S,
    open_extensions=OPEN_EXTENSIONS,
    close_flexions=CLOSE_FLEXIONS,
    end_tolerance=END_TOLERANCE_DEG,
    motion_profile=MOTION_PROFILE,
    side="right",
    spread=15,  # Độ tách hai ngón trong dấu V.
    allow_voltage_warning=ALLOW_VOLTAGE_WARNING,
    dry_run=DRY_RUN,
)
for row in hand.inspect():
    print(
        f"ID {row['id']}: {row['position_deg']:+.2f}°, "
        f"goal={row['goal_deg']:+.2f}°, error={row['error_deg']:+.2f}°, load={row['load']}, "
        f"{row['voltage_V']:.1f} V, {row['temperature_C']} °C, "
        f"torque={row['torque']}, status={row['status']}"
    )
print("Sẵn sàng. Chọn một cell tư thế bên dưới.")

Python đang dùng: c:\Users\Admin\Documents\amazing-hand\.venv\Scripts\python.exe
Helper đang dùng: c:\Users\Admin\Documents\AmazingHand\PythonExample\AmazingHand_Notebook.py
ID 1: -62.70°, goal=-64.16°, error=+1.46°, load=105, 4.7 V, 33 °C, torque=1, status=0
ID 2: +65.33°, goal=+66.80°, error=-1.46°, load=-105, 4.5 V, 34 °C, torque=1, status=0
ID 3: -61.52°, goal=-63.28°, error=+1.76°, load=120, 4.8 V, 34 °C, torque=1, status=0
ID 4: +48.93°, goal=+49.80°, error=-0.88°, load=-75, 4.9 V, 34 °C, torque=1, status=0
ID 5: -38.09°, goal=-39.26°, error=+1.17°, load=90, 4.9 V, 31 °C, torque=1, status=0
ID 6: +40.72°, goal=+41.89°, error=-1.17°, load=-90, 4.6 V, 30 °C, torque=1, status=0
ID 7: -46.00°, goal=-47.17°, error=+1.17°, load=90, 4.6 V, 28 °C, torque=1, status=0
ID 8: +34.28°, goal=+34.86°, error=-0.59°, load=0, 4.6 V, 25 °C, torque=1, status=0
Sẵn sàng. Chọn một cell tư thế bên dưới.


## 1. Duỗi toàn bộ 4 ngón

In [7]:
hand.open_all()

State saved: C:\Users\Admin\Documents\AmazingHand\.tools\hardware\before-grip-cycles-1791536774447028800.json
All four fingers together: custom once at 40 deg/s, then hold the pose.
DIAGNOSTIC: only voltage status bit 0 may pass within 4.0-7.4 V. All other status bits remain fatal. No EEPROM writes.
OPEN_ALL: simultaneous custom, smooth, nominal 7.4 s; pacing follows servo feedback.
OPEN_ALL_CONFIRMED: all eight motors within 1.60 deg of target.
OPEN_ALL_HOLD_CONFIRMED: all eight motors reached the pose and held it. Torque stays enabled. Visually confirm the finger joints.


## 2. Co toàn bộ 4 ngón

In [2]:
hand.close_all()

State saved: C:\Users\Admin\Documents\AmazingHand\.tools\hardware\before-grip-cycles-1791536036899120600.json
All four fingers together: custom once at 40 deg/s, then hold the pose.
DIAGNOSTIC: only voltage status bit 0 may pass within 4.0-7.4 V. All other status bits remain fatal. No EEPROM writes.
CLOSE_ALL: simultaneous custom, smooth, nominal 7.4 s; pacing follows servo feedback.
CLOSE_ALL_CONFIRMED: all eight motors within 3.55 deg of target.
CLOSE_ALL_HOLD_CONFIRMED: all eight motors reached the pose and held it. Torque stays enabled. Visually confirm the finger joints.


## 3. Dấu V / hi sign

Giơ và tách index + middle; ring + thumb co xuống.

In [ ]:
hand.v_sign()

## 4. Chỉ giơ ngón trỏ — index

Ba ngón còn lại co xuống.

In [5]:
hand.raise_finger("index")

State saved: C:\Users\Admin\Documents\AmazingHand\.tools\hardware\before-grip-cycles-1791536147059807700.json
All four fingers together: custom once at 40 deg/s, then hold the pose.
DIAGNOSTIC: only voltage status bit 0 may pass within 4.0-7.4 V. All other status bits remain fatal. No EEPROM writes.
RAISE_INDEX: simultaneous custom, smooth, nominal 7.4 s; pacing follows servo feedback.
RAISE_INDEX_CONFIRMED: all eight motors within 3.55 deg of target.
RAISE_INDEX_HOLD_CONFIRMED: all eight motors reached the pose and held it. Torque stays enabled. Visually confirm the finger joints.


## 5. Chỉ giơ ngón giữa — middle

Ba ngón còn lại co xuống.

In [6]:
hand.raise_finger("middle")

State saved: C:\Users\Admin\Documents\AmazingHand\.tools\hardware\before-grip-cycles-1791536159771707800.json
All four fingers together: custom once at 40 deg/s, then hold the pose.
DIAGNOSTIC: only voltage status bit 0 may pass within 4.0-7.4 V. All other status bits remain fatal. No EEPROM writes.
RAISE_MIDDLE: simultaneous custom, smooth, nominal 7.4 s; pacing follows servo feedback.
RAISE_MIDDLE_CONFIRMED: all eight motors within 2.13 deg of target.
RAISE_MIDDLE_HOLD_CONFIRMED: all eight motors reached the pose and held it. Torque stays enabled. Visually confirm the finger joints.


## 6. Chỉ giơ ngón áp út — ring

Ba ngón còn lại co xuống.

In [ ]:
hand.raise_finger("ring")

## 7. Chỉ giơ ngón cái — thumb

Ba ngón còn lại co xuống.

In [ ]:
hand.raise_finger("thumb")

## Thử chậm riêng index và middle

Bốn cell sau chỉ gửi lệnh tới ngón được chọn, tốc độ 4°/giây, giữ nguyên mục tiêu của các ngón khác. Kết quả trả về góc đặt, góc đo, sai lệch và tải. Chạy một cell mỗi lần để đối chiếu với đốt cuối thực tế.

Nếu muốn chỉnh nhỏ sau khi quan sát, có thể gọi `hand.adjust_endpoint("index", "open", delta=1)` hoặc thay ngón/tư thế thành `middle`/`close`. Mỗi lần chỉ thay tối đa 2°. Chỉ cập nhật cấu hình trong RAM sau khi servo tới mục tiêu; gặp lỗi sẽ giữ cấu hình cũ. Chạy lại cell khởi tạo sẽ dùng lại các tuple phía trên. Các mức đã tới bằng servo cần đối chiếu với tư thế khớp thực tế; không chạy lặp để dò chặn cơ khí.


### Thử duỗi riêng index

In [ ]:
hand.test_finger("index", "open")

### Thử co riêng index

In [ ]:
hand.test_finger("index", "close")

### Thử duỗi riêng middle

In [ ]:
hand.test_finger("middle", "open")

### Thử co riêng middle

In [ ]:
hand.test_finger("middle", "close")

## Nhả lực giữ của toàn bộ servo

Chạy cell này khi muốn ngừng giữ tư thế. Nếu cần ngắt thao tác đang chạy, dùng nút **Interrupt** của Jupyter; script sẽ cố nhả cả 8 servo khi nhận ngắt hoặc gặp lỗi. Cell dưới nhả torque mà không chạy thêm tư thế nào. Với `DRY_RUN=True`, cell này cũng không gửi lệnh.


In [ ]:
hand.release()

## Mở lại notebook

Từ PowerShell tại thư mục project:

```powershell
.\.venv\Scripts\jupyter-lab.exe .\PythonExample\AmazingHand_Control.ipynb
```

Hoặc mở trực tiếp file trong VS Code rồi chọn kernel **AmazingHand (.venv)**. Nếu báo cổng đang bận, dừng chương trình khác đang dùng bàn tay trước khi chạy lại. Nếu một cell báo lỗi, kiểm tra nội dung lỗi và chạy lại cell khởi tạo trước thao tác tiếp theo.
